In [1]:
import csv

import snowflake.connector
import pandas as pd
import configparser

In [2]:
config = configparser.ConfigParser()
config.read('config.ini')
config['snowflake']

<Section: snowflake>

In [12]:
conn = snowflake.connector.connect(
            user= config['snowflake']['user'],
            authenticator = config['snowflake']['authenticator'],
            account=config['snowflake']['account'],
            warehouse=config['snowflake']['warehouse'],
            database=config['snowflake']['db'],
            schema=config['snowflake']['schema'],
            role=config['snowflake']['role']
        )

Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://salesforce.okta.com/app/snowflake/exk17a8fjnwVUjknj697/sso/saml?SAMLRequest=jVJdb9owFP0rkfec2Am0tBZQ0TJUpnZDBVptbyZ2wMSxPV%2Bngf36OXxU3UOrSX6w7HPuOfee27%2FZVSp6FQ6k0QOUJgRFQueGS70eoOViEl%2BhCDzTnCmjxQDtBaCbYR9YpSwd1X6jn8TvWoCPQiENtP0YoNppahhIoJpVAqjP6Xz0%2BECzhFAGIJwPcuhE4SCD1sZ7SzFumiZpOolxa5wRQjC5xgHVQr6gdxL2cw3rjDe5UWfKLvT0gUSKSbeVCIigMDsRb6U%2BjuAzldURBPR%2BsZjFsx%2FzBYpG5%2B7ujIa6Em4u3KvMxfLp4WgAggMoeB5zG1vHE9CmKRQrRW4qW%2FtQLwk3XAiOlVnLMKXpeIBsKfl%2BVH7rdjPf2PnLfZN%2FrVZ%2FiHeb8a6Q1c%2BJvOWsI1fwuL94ITmKns%2BZZm2mU4BaTHWbpA9PJLuISRpnnUWaUdILJyHp5S8UjUOSUjN%2FYL7ZZUpAYVwuElN6dvDHrMVv1rHYlWmPXRVb3Twvt6XeXl73MIDBbVbouC704MEN%2F38Iffyed9q57yGG6XhmlMz30cS4ivmPU0qT9PAieVwcoFRUTKoR504AhLSUMs2dE8yH1fauFggPj6r%2FLvfwLw%3D%3D&RelayState=v

In [13]:
import os
import configparser
import pandas as pd
import snowflake.connector

In [14]:

def load_config(config_path):
    """
    Load Snowflake connection credentials from a config file.
    
    Config file format (snowflake_config.ini):
    [snowflake]
    account = your_account
    username = your_username
    password = your_password
    warehouse = your_warehouse
    database = your_database
    schema = your_schema
    """
    config = configparser.ConfigParser()
    config.read(config_path)
    return config['snowflake']

In [15]:

def connect_to_snowflake(config):
    """Establish connection to Snowflake using config parameters."""
    try:
        conn = snowflake.connector.connect(
            account=config['account'],
            user=config['username'],
            authenticator=config['authenticator'],
            warehouse=config['warehouse'],
            database=config['database'],
            schema=config['schema']
        )
        return conn
    except Exception as e:
        print(f"Error connecting to Snowflake: {e}")
        return None

In [ ]:

def load_csv_to_snowflake(csv_path, table_name, config_path):
    """
    Main function to load CSV to Snowflake table.
    
    Args:
    - csv_path: Path to the local CSV file
    - table_name: Target Snowflake table name
    - config_path: Path to Snowflake configuration file
    """
    # Load configuration
    config = load_config(config_path)
    
    # Connect to Snowflake
    conn = connect_to_snowflake(config)
    if not conn:
        return
    
    try:
        # Create a cursor object
        cursor = conn.cursor()
        
        # Read CSV file
        df = pd.read_csv(csv_path)
        
        # Write DataFrame to Snowflake
        cursor.execute(f"USE WAREHOUSE {config['warehouse']}")
        cursor.execute(f"USE DATABASE {config['database']}")
        cursor.execute(f"USE SCHEMA {config['schema']}")
        
        # Write DataFrame to Snowflake table
        from snowflake.connector.pandas_tools import write_pandas
        success, nchunks, nrows, _ = write_pandas(
            conn, 
            df, 
            table_name.upper()  # Snowflake typically uses uppercase table names
        )
        
        print(f"Successfully loaded {nrows} rows to {table_name}")
    
    except Exception as e:
        print(f"Error loading data to Snowflake: {e}")
    
    finally:
        # Close cursor and connection
        cursor.close()
        conn.close()

In [ ]:

# Example usage
if __name__ == "__main__":
    # Replace with your actual paths
    CSV_PATH = '/path/to/your/input/file.csv'
    CONFIG_PATH = '/path/to/your/snowflake_config.ini'
    TABLE_NAME = 'your_target_table'
    
    load_csv_to_snowflake(CSV_PATH, TABLE_NAME, CONFIG_PATH)

In [16]:
CONNFIG_PATH = 'config.ini'
CSV_PATH = 'C:/Users/djindal/Downloads/ORDERS-JULY24/ProcessedOrders_JUL-2024_1737624885049.csv'

In [30]:
 df = pd.read_csv(
            CSV_PATH, 
            header=0,               # Use first row as header
            delimiter=',',
            skipinitialspace=True,
            # engine='python',         # More flexible parsing
            encoding='utf-8'         # Specify encoding
        )

ParserError: Error tokenizing data. C error: Expected 229 fields in line 8, saw 231
